# LM9000 Aeroderivative Gas Turbine Model — Phase 1

**Objective:** Analyze the GE LM9000 steady-state performance using the newly developed
`LM9000SimpleCycle` and `LM9000CombinedCycle` models calibrated to GE datasheet specifications.

## Key Specs (from GE datasheet)

| Configuration | Net Power | Efficiency | Heat Rate | Exhaust Temp |
|---------------|-----------|-----------|-----------|---------------|
| Simple Cycle (Power Gen) | 56.723 MW | 39.52% | 9,109 kJ/kW-hr | 456°C (729 K) |
| Combined Cycle (Power Gen) | 72.471 MW | 50.48% | 7,132 kJ/kW-hr | 380°C (stack) |

## Component Configuration

- **4-stage low-pressure compressor (LPC)** — PR ~3.8, η ~88%
- **9-stage high-pressure compressor (HPC)** — PR ~5.2, η ~86%
- **DLE 1.5 combustor** — 15 ppm NOx, ~98% combustion efficiency
- **2-stage high-pressure turbine (HPT)** — drives HPC, η ~91%
- **1-stage intermediate-pressure turbine (IPT)** — drives LPC, η ~88%
- **4-stage free power turbine (FPT)** — drives generator/load, η ~90%

## Approach

The models are built from thermodynamic first principles, calibrated to the datasheet
anchor points. Part-load behavior uses polynomial efficiency curves fitted to real
aeroderivative characteristics.

**End goal:** Use these models as prime movers in transient islanding simulations
coupled with ANDES for electrical dynamics.

In [ ]:
import sys
from pathlib import Path

# Add project root so gas_plant is importable
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gas_plant import LM9000SimpleCycle, LM9000CombinedCycle, Fleet

print(f"Project root: {ROOT}")
print(f"LM9000 models imported successfully")

## 1. Instantiate LM9000 Models

Create both simple-cycle and combined-cycle variants.

In [ ]:
# Simple-cycle model: 56.723 MW, 39.52% efficiency
lm9000_sc = LM9000SimpleCycle()

# Combined-cycle model: 72.471 MW, 50.48% efficiency
lm9000_cc = LM9000CombinedCycle()

print(lm9000_sc)
print(lm9000_cc)

## 2. Full-Load Validation Against GE Datasheet

Compare model outputs at design point (load=1.0) against the provided specifications.

In [ ]:
# Dispatch at full load
sc_full = lm9000_sc.dispatch(1.0)
cc_full = lm9000_cc.dispatch(1.0)

# GE datasheet targets
SC_POWER_MW = 56.723
SC_EFFICIENCY = 0.3952
SC_EXHAUST_T_K = 729.0  # 456°C
SC_HEAT_RATE = 9109.0  # kJ/kWh

CC_POWER_MW = 72.471
CC_EFFICIENCY = 0.5048
CC_HEAT_RATE = 7132.0  # kJ/kWh

print("\n" + "="*70)
print("SIMPLE CYCLE — Full Load Validation")
print("="*70)
print(f"{'Parameter':<25} {'Model':<15} {'Datasheet':<15} {'Error':<10}")
print("-"*70)

sc_power_mw = sc_full["power_w"] / 1e6
sc_eff = sc_full["efficiency"]
sc_hr = 3600.0 / sc_eff if sc_eff > 0 else float("inf")
sc_exh_t = sc_full["exhaust_T_K"]

print(f"{'Power [MW]':<25} {sc_power_mw:<15.2f} {SC_POWER_MW:<15.3f} {(sc_power_mw-SC_POWER_MW)/SC_POWER_MW*100:+.2f}%")
print(f"{'Efficiency [-]':<25} {sc_eff:<15.4f} {SC_EFFICIENCY:<15.4f} {(sc_eff-SC_EFFICIENCY)/SC_EFFICIENCY*100:+.2f}%")
print(f"{'Heat Rate [kJ/kWh]':<25} {sc_hr:<15.0f} {SC_HEAT_RATE:<15.0f} {(sc_hr-SC_HEAT_RATE)/SC_HEAT_RATE*100:+.2f}%")
print(f"{'Exhaust T [K]':<25} {sc_exh_t:<15.0f} {SC_EXHAUST_T_K:<15.0f} {sc_exh_t-SC_EXHAUST_T_K:+.0f} K")
print(f"{'Fuel flow [kg/s]':<25} {sc_full['fuel_kg_s']:<15.3f}")
print(f"{'Exhaust flow [kg/s]':<25} {sc_full['exhaust_m_kg_s']:<15.1f}")
print(f"{'CO2 [kg/MWh]':<25} {(sc_full['co2_kg_s'] * 3600) / sc_power_mw:<15.1f}")

print("\n" + "="*70)
print("COMBINED CYCLE — Full Load Validation")
print("="*70)
print(f"{'Parameter':<25} {'Model':<15} {'Datasheet':<15} {'Error':<10}")
print("-"*70)

cc_power_mw = cc_full["power_w"] / 1e6
cc_eff = cc_full["efficiency"]
cc_hr = 3600.0 / cc_eff if cc_eff > 0 else float("inf")

print(f"{'Power [MW]':<25} {cc_power_mw:<15.2f} {CC_POWER_MW:<15.3f} {(cc_power_mw-CC_POWER_MW)/CC_POWER_MW*100:+.2f}%")
print(f"{'Efficiency [-]':<25} {cc_eff:<15.4f} {CC_EFFICIENCY:<15.4f} {(cc_eff-CC_EFFICIENCY)/CC_EFFICIENCY*100:+.2f}%")
print(f"{'Heat Rate [kJ/kWh]':<25} {cc_hr:<15.0f} {CC_HEAT_RATE:<15.0f} {(cc_hr-CC_HEAT_RATE)/CC_HEAT_RATE*100:+.2f}%")
print(f"{'Fuel flow [kg/s]':<25} {cc_full['fuel_kg_s']:<15.3f}")
print(f"{'Efficiency gain [pp]':<25} {(cc_eff - sc_eff)*100:<15.2f}")

## 3. Part-Load Performance Sweep

Dispatch across the full load range [0, 1] to examine part-load behavior.

In [ ]:
# Sweep load from 0 to 1 in 21 steps
loads = np.linspace(0, 1, 21)
sc_results = lm9000_sc.dispatch(loads)
cc_results = lm9000_cc.dispatch(loads)

# Build DataFrames
sc_df = pd.DataFrame({
    'load_frac': loads,
    'power_mw': sc_results['power_w'] / 1e6,
    'fuel_kg_s': sc_results['fuel_kg_s'],
    'efficiency': sc_results['efficiency'],
    'exhaust_T_K': sc_results['exhaust_T_K'],
    'exhaust_m_kg_s': sc_results['exhaust_m_kg_s'],
})

cc_df = pd.DataFrame({
    'load_frac': loads,
    'power_mw': cc_results['power_w'] / 1e6,
    'fuel_kg_s': cc_results['fuel_kg_s'],
    'efficiency': cc_results['efficiency'],
    'exhaust_T_K': cc_results['exhaust_T_K'],
    'exhaust_m_kg_s': cc_results['exhaust_m_kg_s'],
})

# Derived: heat rate
sc_df['heat_rate_btu_kwh'] = np.where(
    sc_df['efficiency'] > 0,
    3412.0 / sc_df['efficiency'],
    np.inf
)

cc_df['heat_rate_btu_kwh'] = np.where(
    cc_df['efficiency'] > 0,
    3412.0 / cc_df['efficiency'],
    np.inf
)

print("\nSimple-Cycle Part-Load Performance")
print(sc_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n\nCombined-Cycle Part-Load Performance")
print(cc_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

## 4. Part-Load Characteristic Plots

Visualize power, efficiency, fuel flow, and exhaust temperature vs. load fraction.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Power
ax = axes[0, 0]
ax.plot(sc_df['load_frac'] * 100, sc_df['power_mw'], 'b-o', markersize=4, label='Simple Cycle')
ax.plot(cc_df['load_frac'] * 100, cc_df['power_mw'], 'r-s', markersize=4, label='Combined Cycle')
ax.axhline(56.723, color='b', ls='--', lw=1, alpha=0.5, label='SC Rated (56.7 MW)')
ax.axhline(72.471, color='r', ls='--', lw=1, alpha=0.5, label='CC Rated (72.5 MW)')
ax.set_xlabel('Load [%]')
ax.set_ylabel('Power [MW]')
ax.set_title('LM9000 Electrical Power Output')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Efficiency
ax = axes[0, 1]
ax.plot(sc_df['load_frac'] * 100, sc_df['efficiency'] * 100, 'b-o', markersize=4, label='Simple Cycle')
ax.plot(cc_df['load_frac'] * 100, cc_df['efficiency'] * 100, 'r-s', markersize=4, label='Combined Cycle')
ax.axhline(39.52, color='b', ls='--', lw=1, alpha=0.5)
ax.axhline(50.48, color='r', ls='--', lw=1, alpha=0.5)
ax.set_xlabel('Load [%]')
ax.set_ylabel('Efficiency [%]')
ax.set_title('LM9000 Thermal Efficiency')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Fuel flow
ax = axes[1, 0]
ax.plot(sc_df['load_frac'] * 100, sc_df['fuel_kg_s'], 'b-o', markersize=4, label='Simple Cycle')
ax.plot(cc_df['load_frac'] * 100, cc_df['fuel_kg_s'], 'r-s', markersize=4, label='Combined Cycle')
ax.set_xlabel('Load [%]')
ax.set_ylabel('Fuel Flow [kg/s]')
ax.set_title('Fuel Consumption')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Exhaust temperature
ax = axes[1, 1]
ax.plot(sc_df['load_frac'] * 100, sc_df['exhaust_T_K'], 'b-o', markersize=4, label='Simple Cycle (GT exit)')
ax.plot(cc_df['load_frac'] * 100, cc_df['exhaust_T_K'], 'r-s', markersize=4, label='Combined Cycle (HRSG stack)')
ax.axhline(729, color='b', ls='--', lw=1, alpha=0.5)
ax.axhline(380, color='r', ls='--', lw=1, alpha=0.5)
ax.set_xlabel('Load [%]')
ax.set_ylabel('Temperature [K]')
ax.set_title('Exhaust Temperature')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ROOT / 'examples' / 'output' / 'lm9000_part_load.png', dpi=110, bbox_inches='tight')
print(f"Plot saved to examples/output/lm9000_part_load.png")
plt.show()

## 5. Fleet Integration

Demonstrate that LM9000 models work seamlessly with other plant types in a Fleet.

In [ ]:
# Create a mixed fleet: 1 LM9000 SC + 1 LM9000 CC
fleet = Fleet([lm9000_sc, lm9000_cc])

# Dispatch at varied loads
fleet_loads = np.array([0.5, 0.75])
fleet_result = fleet.dispatch(fleet_loads)

print("\nFleet Dispatch (LM9000 SC + LM9000 CC)")
print(f"Unit 1 (SC) load: {fleet_loads[0]:.2f} => {fleet_result['units'][0]['power_w']/1e6:.2f} MW")
print(f"Unit 2 (CC) load: {fleet_loads[1]:.2f} => {fleet_result['units'][1]['power_w']/1e6:.2f} MW")
print(f"Total fleet power: {fleet_result['power_w']/1e6:.2f} MW")
print(f"Fleet efficiency: {fleet_result['efficiency']:.4f}")

## 6. Emissions Summary

Examine CO2 and specific emissions across the operating range.

In [ ]:
# Add specific CO2 emissions to dataframes
sc_df['co2_kg_mwh'] = np.where(
    sc_df['power_mw'] > 0,
    (sc_df['fuel_kg_s'] * lm9000_sc.co2_per_fuel_kg * 3600) / sc_df['power_mw'],
    0
)

cc_df['co2_kg_mwh'] = np.where(
    cc_df['power_mw'] > 0,
    (cc_df['fuel_kg_s'] * lm9000_cc.co2_per_fuel_kg * 3600) / cc_df['power_mw'],
    0
)

print("\nCO2 Emissions Comparison")
print(f"{'Load':<8} {'SC CO2 [kg/MWh]':<20} {'CC CO2 [kg/MWh]':<20} {'Reduction':<15}")
print("-" * 65)
for i in [0, 5, 10, 15, 20]:
    sc_co2 = sc_df.iloc[i]['co2_kg_mwh']
    cc_co2 = cc_df.iloc[i]['co2_kg_mwh']
    reduction = (1 - cc_co2/sc_co2) * 100 if sc_co2 > 0 else 0
    load = sc_df.iloc[i]['load_frac'] * 100
    print(f"{load:<8.0f} {sc_co2:<20.1f} {cc_co2:<20.1f} {reduction:<14.1f}%")

print(f"\nFull-load datasheet values:")
print(f"  Simple Cycle: 492.9 kg CO2/MWh (model: {sc_df.iloc[-1]['co2_kg_mwh']:.1f})")
print(f"  Combined Cycle: 383.6 kg CO2/MWh (model: {cc_df.iloc[-1]['co2_kg_mwh']:.1f})")